In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re


DATA_PATH = "/Users/danylo/Desktop/coursework/lab5/data/processed_v2.csv" 

TRAIN_IDS_PATH = "/Users/danylo/Desktop/coursework/lab5/data/sample/splits_train_ids.txt"
VAL_IDS_PATH = "/Users/danylo/Desktop/coursework/lab5/data/sample/splits_val_ids.txt"
TEST_IDS_PATH = "/Users/danylo/Desktop/coursework/lab5/data/sample/splits_test_ids.txt"

print("Завантаження даних...")
df = pd.read_csv(DATA_PATH)

if 'id' not in df.columns:
    df['id'] = [f"nli_{i}" for i in range(len(df))]

def load_ids(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Файл {filepath} не знайдено. Спочатку запусти split.py!")
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f.readlines()]

print("Відновлення вибірок (Train/Val/Test) за ID...")
train_ids = load_ids(TRAIN_IDS_PATH)
val_ids = load_ids(VAL_IDS_PATH)
test_ids = load_ids(TEST_IDS_PATH)

train_df = df[df['id'].isin(train_ids)].copy()
val_df = df[df['id'].isin(val_ids)].copy()
test_df = df[df['id'].isin(test_ids)].copy()

print(f"Розміри вибірок: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}\n")


print("--- 1.3 Перевірки балансу ---")
print("Розподіл класів (у відсотках):")
for name, d in zip(['Train', 'Val', 'Test'], [train_df, val_df, test_df]):
    print(f"{name}:")
    print(d['label_name'].value_counts(normalize=True).round(3).to_string())
    print("-" * 20)

print("\nРозподіл довжин текстів (premise + hypothesis):")
for name, d in zip(['Train', 'Val', 'Test'], [train_df, val_df, test_df]):
    lens = (d['premise_v2'].astype(str) + " " + d['hypothesis_v2'].astype(str)).str.len()
    print(f"{name} -> Mean: {lens.mean():.1f}, Median: {lens.median()}, 95th: {lens.quantile(0.95):.1f}")

print("\n" + "="*40 + "\n")


print("--- 2. Перевірки на Leakage ---")

def get_combined_texts(d):
    return set((d['premise_v2'].astype(str) + " [SEP] " + d['hypothesis_v2'].astype(str)).str.lower().str.strip())

train_texts = get_combined_texts(train_df)
val_texts = get_combined_texts(val_df)
test_texts = get_combined_texts(test_df)

print("2.1 Exact duplicates (Точні дублікати):")
print(f"  train ∩ test = {len(train_texts.intersection(test_texts))}")
print(f"  train ∩ val  = {len(train_texts.intersection(val_texts))}")

print("\n2.2 Near-duplicate leakage (Косинусна схожість > 0.95):")
vectorizer = TfidfVectorizer(max_features=5000)

train_docs = (train_df['premise_v2'].astype(str) + " " + train_df['hypothesis_v2'].astype(str)).tolist()
test_docs = (test_df['premise_v2'].astype(str) + " " + test_df['hypothesis_v2'].astype(str)).tolist()

print("  Обчислення TF-IDF матриць...")
train_tfidf = vectorizer.fit_transform(train_docs)
test_tfidf = vectorizer.transform(test_docs)

print("  Обчислення косинусної схожості...")
sim_matrix = cosine_similarity(test_tfidf, train_tfidf)

high_sim_pairs = np.where((sim_matrix > 0.95) & (sim_matrix < 0.999))
print(f"  Знайдено near-duplicates між Train та Test: {len(high_sim_pairs[0])}")

print("\n2.3 Template / metadata leakage:")
suspicious_patterns = r"(label=|Category:|class=|topic=|entailment|contradiction|neutral)"
leak_mask = test_df['hypothesis_v2'].astype(str).str.contains(suspicious_patterns, flags=re.IGNORECASE, na=False)
leak_count = leak_mask.sum()
print(f"  Знайдено підозрілих метаданих у текстах: {leak_count}")
if leak_count > 0:
    print("  Приклади:")
    print(test_df[leak_mask]['hypothesis_v2'].head(3).to_string())

print("\n2.4 Group leakage (Спільні передумови):")
train_premises = set(train_df['premise_v2'].astype(str).str.lower().str.strip())
test_premises = set(test_df['premise_v2'].astype(str).str.lower().str.strip())
group_leakage = train_premises.intersection(test_premises)
print(f"  Group Leakage (спільні premise у Train та Test): {len(group_leakage)}")
if len(group_leakage) == 0:
    print("передумови не перетинаються!")
else:
    print("перетин груп")


Завантаження даних...
Відновлення вибірок (Train/Val/Test) за ID...
Розміри вибірок: Train=1599, Val=201, Test=200

--- 1.3 Перевірки балансу ---
Розподіл класів (у відсотках):
Train:
label_name
contradiction    0.346
entailment       0.338
neutral          0.316
--------------------
Val:
label_name
contradiction    0.373
neutral          0.313
entailment       0.313
--------------------
Test:
label_name
entailment       0.345
contradiction    0.330
neutral          0.325
--------------------

Розподіл довжин текстів (premise + hypothesis):
Train -> Mean: 98.2, Median: 91.0, 95th: 165.0
Val -> Mean: 99.6, Median: 94.0, 95th: 157.0
Test -> Mean: 95.3, Median: 89.0, 95th: 155.2


--- 2. Перевірки на Leakage ---
2.1 Exact duplicates (Точні дублікати):
  train ∩ test = 0
  train ∩ val  = 0

2.2 Near-duplicate leakage (Косинусна схожість > 0.95):
  Обчислення TF-IDF матриць...
  Обчислення косинусної схожості...
  Знайдено near-duplicates між Train та Test: 0

2.3 Template / metadata leakag

/var/folders/pw/62qh19vj00sc6dr9twn8k0yr0000gp/T/ipykernel_96764/4288240885.py:86: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  leak_mask = test_df['hypothesis_v2'].astype(str).str.contains(suspicious_patterns, flags=re.IGNORECASE, na=False)
